# Use custom software spec to create `statsmodels` function describing data with `ibm-watsonx-ai`

This notebook demonstrates how to deploy in watsonx.ai Runtime service a python function with `statsmodel` which requires to create custom software specification using conda yaml file with all required libraries.  
Some familiarity with bash is helpful. This notebook uses Python 3.12.


## Learning goals

The learning goals of this notebook are:

-  Working with the watsonx.ai Runtime instance
-  Creating custom software specification
-  Online deployment of python function
-  Scoring data using deployed function

## Contents

This notebook contains the following parts:

1. [Set up the environment](#1.-Set-up-the-environment)
2. [Create function](#2.-Create-function)
3. [Upload python function](#3.-Upload-python-function)
4. [Create online deployment](#4.-Create-online-deployment)
5. [Scoring](#5.-Scoring)
6. [Cleanup](#6.-Cleanup)
7. [Summary and next steps](#7.-Summary-and-next-steps)

<a id="1.-Set-up-the-environment"></a>
## 1. Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).

### Install and import the `ibm-watsonx-ai` and dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U ibm-watsonx-ai | tail -n 1
%pip install statsmodels | tail -n 1

### Connection to watsonx.ai Runtime

Authenticate the watsonx.ai Runtime service on IBM Cloud. You need to provide platform `api_key` and instance `location`.

You can use [IBM Cloud CLI](https://cloud.ibm.com/docs/cli/index.html) to retrieve platform API Key and instance location.

API Key can be generated in the following way:
```
ibmcloud login
ibmcloud iam api-key-create API_KEY_NAME
```

In result, get the value of `api_key` from the output.


Location of your watsonx.ai Runtime instance can be retrieved in the following way:
```
ibmcloud login --apikey API_KEY -a https://cloud.ibm.com
ibmcloud resource service-instance INSTANCE_NAME
```

In result, get the value of `location` from the output.

**Tip**: Your `Cloud API key` can be generated by going to the [**Users** section of the Cloud console](https://cloud.ibm.com/iam#/users). From that page, click your name, scroll down to the **API Keys** section, and click **Create an IBM Cloud API key**. Give your key a name and click **Create**, then copy the created key and paste it below. You can also get a service specific url by going to the [**Endpoint URLs** section of the watsonx.ai Runtime docs](https://cloud.ibm.com/apidocs/machine-learning).  You can check your instance location in your  <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance details.

You can also get service specific apikey by going to the [**Service IDs** section of the Cloud Console](https://cloud.ibm.com/iam/serviceids).  From that page, click **Create**, then copy the created key and paste it below.

**Action**: Enter your `url` and `api_key` in the following cell.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
)

In [3]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

First, create a space that will be used for your work. If you do not have space already created, you can use [Deployment Spaces Dashboard](https://dataplatform.cloud.ibm.com/ml-runtime/spaces?context=cpdaas) to create one.

- Click New Deployment Space
- Create an empty space
- Select Cloud Object Storage
- Select watsonx.ai Runtime instance and press Create
- Copy `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: Assign space ID below

In [4]:
space_id = "PASTE YOUR SPACE ID HERE"

You can use `list` method to print all existing spaces.

In [ ]:
client.spaces.list(limit=10)

To be able to interact with all resources available in watsonx.ai Runtime, you need to set **space** which you will be using.

In [5]:
client.set.default_space(space_id)

'SUCCESS'

<a id="2.-Create-function"></a>
## 2. Create function

In this section you will learn how to create deployable function with statsmodels module calculating describition of a given data.

#### Create deploayable callable which uses stsmodels library

In [6]:
def deployable_callable():
    """
    Deployable python function with score
    function implemented.
    """
    try:
        from statsmodels.stats.descriptivestats import describe
    except ModuleNotFoundError as e:
        print(f"statsmodels not installed: {str(e)}")

    def score(payload):
        """
        Score method.
        """
        try:
            data = payload["input_data"][0]["values"]
            return {"predictions": [{"values": str(describe(data))}]}
        except Exception as e:
            return {"predictions": [{"values": [repr(e)]}]}

    return score

####  Test callable locally

In [7]:
import numpy as np

score_function = deployable_callable()

data = np.random.randn(10, 10)
data_description = score_function({"input_data": [{"values": data}]})

print(data_description["predictions"][0]["values"])

                          0          1          2          3          4  \
nobs              10.000000  10.000000  10.000000  10.000000  10.000000   
missing            0.000000   0.000000   0.000000   0.000000   0.000000   
mean              -0.142165  -0.071109  -0.071099   0.076253  -0.518539   
std_err            0.242244   0.308282   0.346994   0.340890   0.240426   
upper_ci           0.332625   0.533114   0.608997   0.744386  -0.047313   
lower_ci          -0.616955  -0.675331  -0.751196  -0.591879  -0.989765   
std                0.766043   0.974874   1.097293   1.077989   0.760294   
iqr                0.944193   1.125893   1.290023   1.552004   0.551361   
iqr_normal         0.699931   0.834626   0.956296   1.150502   0.408724   
mad                0.593256   0.815466   0.846527   0.929699   0.581861   
mad_normal         0.743537   1.022035   1.060964   1.165205   0.729255   
coef_var          -5.388403 -13.709627 -15.433254  14.136924  -1.466223   
range              2.1964

<a id="3.-Upload-python-function"></a>
## 3. Upload python function

In this section you will learn how to upload the python function to the Cloud.

#### Custom software_specification
Create new software specification based on default Python 3.12 environment extended by statsmodels package.

In [8]:
requirements_txt_content = "statsmodels"

with open("requirements.txt", "w") as file:
    file.write(requirements_txt_content)

In [9]:
base_sw_spec_id = client.software_specifications.get_id_by_name("runtime-25.1-py3.12")

The `requirements.txt` file describes details of package extension. Now you need to store new package extension using `APIClient`.

In [10]:
meta_prop_pkg_extn = {
    client.package_extensions.ConfigurationMetaNames.NAME: "statsmodels env",
    client.package_extensions.ConfigurationMetaNames.DESCRIPTION: "Environment with statsmodels",
    client.package_extensions.ConfigurationMetaNames.TYPE: "requirements_txt",
}

pkg_extn_details = client.package_extensions.store(
    meta_props=meta_prop_pkg_extn, file_path="requirements.txt"
)
pkg_extn_id = client.package_extensions.get_id(pkg_extn_details)
pkg_extn_url = client.package_extensions.get_href(pkg_extn_details)

Creating package extension
SUCCESS


#### Create new software specification and add created package extention to it.

In [11]:
meta_prop_sw_spec = {
    client.software_specifications.ConfigurationMetaNames.NAME: "statsmodels software_spec",
    client.software_specifications.ConfigurationMetaNames.DESCRIPTION: "Software specification for statsmodels",
    client.software_specifications.ConfigurationMetaNames.BASE_SOFTWARE_SPECIFICATION: {
        "guid": base_sw_spec_id
    },
}

sw_spec_details = client.software_specifications.store(meta_props=meta_prop_sw_spec)
sw_spec_id = client.software_specifications.get_id(sw_spec_details)

client.software_specifications.add_package_extension(sw_spec_id, pkg_extn_id)

SUCCESS


'SUCCESS'

#### Get the details of created software specification

In [12]:
import json

print(json.dumps(client.software_specifications.get_details(sw_spec_id), indent=2))

#### Store the function

In [13]:
meta_props = {
    client.repository.FunctionMetaNames.NAME: "statsmodels function",
    client.repository.FunctionMetaNames.SOFTWARE_SPEC_ID: sw_spec_id,
}

function_details = client.repository.store_function(
    meta_props=meta_props, function=deployable_callable
)
function_id = client.repository.get_function_id(function_details)

#### Get function details

In [14]:
print(json.dumps(client.repository.get_details(function_id), indent=2))

{
  "metadata": {
    "name": "statsmodels function",
    "space_id": "fb3d528a-bf16-460e-bcf6-06f05d8ba57c",
    "resource_key": "3d1f35e1-9750-4dee-8d73-df73908b47c9",
    "id": "96cfe121-3088-4550-aae9-fe7a53a99b70",
    "created_at": "2026-01-15T10:24:23Z",
    "rov": {
      "member_roles": {
        "IBMid-696000GJGB": {
          "user_iam_id": "IBMid-696000GJGB",
          "roles": [
            "OWNER"
          ]
        }
      }
    },
    "owner": "IBMid-696000GJGB"
  },
  "entity": {
    "software_spec": {
      "id": "cb1dd3a0-2d7a-42ae-953d-06d24dccdf3c"
    },
    "type": "python"
  }
}


**Note:** You can see that function is successfully stored in watsonx.ai Runtime Service.

In [15]:
client.repository.list_functions()

,ID,NAME,CREATED,TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,96cfe121-3088-4550-aae9-fe7a53a99b70,statsmodels function,2026-01-15T10:24:23Z,python,,


<a id="4.-Create-online-deployment"></a>
## 4. Create online deployment
You can use commands bellow to create online deployment for stored function (web service).

#### Create online deployment of a python function

In [16]:
metadata = {
    client.deployments.ConfigurationMetaNames.NAME: "Deployment of statsmodels function",
    client.deployments.ConfigurationMetaNames.ONLINE: {},
}

function_deployment = client.deployments.create(function_id, meta_props=metadata)



######################################################################################

Synchronous deployment creation for id: '96cfe121-3088-4550-aae9-fe7a53a99b70' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
................................
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='496c75d6-6483-4289-956d-36e418b8b937'
-----------------------------------------------------------------------------------------------




In [17]:
client.deployments.list()

,ID,NAME,STATE,CREATED,ARTIFACT_TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,496c75d6-6483-4289-956d-36e418b8b937,Deployment of statsmodels function,ready,2026-01-15T10:24:39.458Z,function,,


Get deployment id.

In [18]:
deployment_id = client.deployments.get_id(function_deployment)
print(deployment_id)

496c75d6-6483-4289-956d-36e418b8b937


<a id="5.-Scoring"></a>
## 5. Scoring

You can send new scoring records to web-service deployment using `score` method.

In [19]:
scoring_payload = {"input_data": [{"values": data}]}

In [20]:
predictions = client.deployments.score(deployment_id, scoring_payload)
print(data_description["predictions"][0]["values"])

                          0          1          2          3          4  \
nobs              10.000000  10.000000  10.000000  10.000000  10.000000   
missing            0.000000   0.000000   0.000000   0.000000   0.000000   
mean              -0.142165  -0.071109  -0.071099   0.076253  -0.518539   
std_err            0.242244   0.308282   0.346994   0.340890   0.240426   
upper_ci           0.332625   0.533114   0.608997   0.744386  -0.047313   
lower_ci          -0.616955  -0.675331  -0.751196  -0.591879  -0.989765   
std                0.766043   0.974874   1.097293   1.077989   0.760294   
iqr                0.944193   1.125893   1.290023   1.552004   0.551361   
iqr_normal         0.699931   0.834626   0.956296   1.150502   0.408724   
mad                0.593256   0.815466   0.846527   0.929699   0.581861   
mad_normal         0.743537   1.022035   1.060964   1.165205   0.729255   
coef_var          -5.388403 -13.709627 -15.433254  14.136924  -1.466223   
range              2.1964

<a id="6.-Cleanup"></a>
## 6. Cleanup

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

see the steps in this sample [notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="7.-Summary-and-next-steps"></a>
## 7. Summary and next steps

 You successfully completed this notebook! You learned how to use watsonx.ai Runtime for function deployment and scoring with custom software_spec.  
 Check out our [Online Documentation](https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/welcome-main.html?context=wx) for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors

**Jan Sołtysik**, Software Engineer Intern at watsonx.ai.

**Rafał Chrzanowski**, Software Engineer at watsonx.ai.

Copyright © 2020-2026 IBM. This notebook and its source code are released under the terms of the MIT License.